# Review and edit a plasmid annotation

Use the same workspace that drives the Sequence tab to select a feature, rename it and reopen the saved edit. The public SB2CL plasmid pIAKA1s271_11_L30_01 has a feature that crosses the origin of its circular map. [GenBank example](../../databases/plasmids/maps/pIAKA1s271_11_L30_01.gb)

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import sys

root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "strandfolio" / "__init__.py").is_file())
sys.path.insert(0, str(root))

from strandfolio.application.workspace import Workspace
from strandfolio.io.formats import readSequence
from strandfolio.operations.editing import editAnnotation, scanLabels
from strandfolio.storage.feature_catalogue import FeatureCatalogue
from strandfolio.storage.projects import ProjectRepository

## Open a saved record

Each run creates a project under `_outputs/U07/`. Save the imported GenBank record there as the parent of the edited copy.

In [ ]:
runId = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output = root / "docs/notebooks/_outputs/U07" / runId
output.mkdir(parents=True)
project = ProjectRepository.create(output / "project")

source = root / "databases/plasmids/maps/pIAKA1s271_11_L30_01.gb"
capture = readSequence(source)
parent = project.saveRevision(capture.records[0], "SB2CL original", capture=capture, requestId="import")
workspace = Workspace(project)
document = workspace.openSavedRevision(parent.revisionId)
print(f"{source.stem} · {len(parent):,} bp · {len(parent.features)} features")

## Review the labels and copy a whole feature

The released catalogue recognizes two annotations in this record. The KanR feature has two intervals because it crosses the origin. Selecting the whole feature joins both intervals in their recorded order. Python intervals start at zero and exclude their end. [Feature catalogue](../../databases/features/README.md)

In [ ]:
with FeatureCatalogue(root / "databases/features/features.sqlite3") as catalogue:
    review = scanLabels(document.revision, catalogue)

for hit in review.suggestions:
    print(hit.label, hit.status, hit.parts)

In [ ]:
wrappedHit = next(hit for hit in review.suggestions if len(hit.parts) == 2)
featureId, = wrappedHit.matchedFeatureIds
session = document.session
session.selectWholeFeature(featureId)
print(session.selection.segments)
print(f"Selected text spans {len(session.selectedText)} bases.")
assert session.selection.wrapsOrigin(len(parent)) and len(session.selectedText) == 816

## Rename, undo and redo

Give the existing annotation a shorter display label. Each action asks the workspace for a target bound to the current revision of the open document.

In [ ]:
def currentTarget():
    return workspace.targetFor(document.documentId)

workspace.applyAnnotationEdit(
    currentTarget(),
    lambda revision, recordId: editAnnotation(revision, featureId, label="KanR", recordId=recordId),
)
print("Edited:", document.revision.getFeature(featureId).label)

workspace.undoAnnotationEdit(currentTarget())
print("After undo:", document.revision.getFeature(featureId).label)

workspace.redoAnnotationEdit(currentTarget())
print("After redo:", document.revision.getFeature(featureId).label)

## Reopen the parent and the edited copy

The two named revisions keep their own labels and share the same nucleotide sequence. The printed project directory can also be opened in the desktop app.

In [ ]:
saved = workspace.saveDocument(currentTarget(), "SB2CL reviewed")
reopened = ProjectRepository.open(project.path)
original = reopened.getRevision(parent.revisionId)
reviewed = reopened.getRevision(saved.revisionId)

for record in (original, reviewed):
    print(record.name, "·", record.getFeature(featureId).label)
assert reviewed.getFeature(featureId).label == "KanR" and original.sequence == reviewed.sequence
print(reopened.path.relative_to(root))